# REMD MBAR Free Energy Analysis

Based on MBAR_Parameter_Analysis-v2.ipynb

This notebook performs free energy surface (FES) analysis using the MBAR method.

In [ ]:
"""
REMD MBAR FES Analysis Notebook
Author: Song Yang
Date: 2025
"""

import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from pathlib import Path

# Import FESAnalyzer from CTGoMartini
from ctgomartini.analysis.remd_mbar import FESAnalyzer

# Set plotting style
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print('Imports successful!')

## 1. Setup and ConfigurationDefine input files and analysis parameters.

In [ ]:
# Configuration
OUTPUT_FILE = 'output.nc'
CV_FILE = './dRMStraj_nc_StateA.dat'  # or StateB
INTERVAL = 5  # CV data interval

# Analysis parameters
ANALYSIS_PARAMS = {
    'g': 1,
    'length_ratio': 1.0,
    'start_point': None,
    'start_ratio': 0.2,
    'selected_state': 5
}

# CV bounds for barrier search
LEFT_BOUND = 35
RIGHT_BOUND = 47

# Output directories
os.makedirs('AnalysisParameter', exist_ok=True)
os.makedirs('FitEnergy', exist_ok=True)

print(f'Output file: {OUTPUT_FILE}')
print(f'CV file: {CV_FILE}')

## 2. Initialize AnalyzerLoad simulation data and initialize FES analyzer.

In [ ]:
# Initialize analyzer
analyzer = FESAnalyzer(OUTPUT_FILE, CV_FILE, interval=INTERVAL)

print(f'Number of states: {analyzer.n_states}')
print(f'Temperatures: {analyzer.temperatures_k}')
print(f'CV shape: {analyzer.cv_values_replica.shape}')

## 3. Single State AnalysisAnalyze free energy surface for a single state.

In [ ]:
# Initialize FES
analyzer.initialize_fes(
    g=ANALYSIS_PARAMS['g'],
    length_ratio=ANALYSIS_PARAMS['length_ratio'],
    start_ratio=ANALYSIS_PARAMS['start_ratio']
)

# Analyze single state
results = analyzer.analyze_onestate(
    selected_state=ANALYSIS_PARAMS['selected_state'],
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)

print('Metrics:')
for key, value in results['metrics'].items():
    print(f'  {key}: {value:.4f}')

In [ ]:
# Plot PMF
fig, ax = plt.subplots(figsize=(10, 6))

cv_values = results['cv_values']
pmf = results['pmf']
pmf_uncertainty = results['pmf_uncertainty']
metrics = results['metrics']

pmf_normalized = pmf - pmf.min()

ax.plot(cv_values, pmf_normalized, 'b-', linewidth=2, label=f"State {ANALYSIS_PARAMS['selected_state']}")
ax.fill_between(cv_values, 
                pmf_normalized - pmf_uncertainty, 
                pmf_normalized + pmf_uncertainty, 
                alpha=0.3, color='b')

ax.axvline(metrics['barrier_pos'], color='r', linestyle='--', label=f"Barrier: {metrics['barrier_pos']:.2f}")
ax.axvline(metrics['basin1_pos'], color='g', linestyle=':', alpha=0.7)
ax.axvline(metrics['basin2_pos'], color='g', linestyle=':', alpha=0.7)

ax.set_xlabel('CV (Å)')
ax.set_ylabel('Free Energy (kJ/mol)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"Barrier: {metrics['barrier']:.2f} kJ/mol, Keq: {metrics['keq']:.4f}")

## 4. Parameter Sweep AnalysisCheck convergence and sensitivity.

In [ ]:
# Start ratio sweep
print('Start ratio sweep...')
start_ratio_results = analyzer.parameter_sweep(
    'start_ratio', 
    [0, 0.1, 0.2, 0.3, 0.4, 0.5],
    default_params=ANALYSIS_PARAMS,
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)
analyzer.save_results(start_ratio_results, 'AnalysisParameter/start_ratio_results.pkl')

In [ ]:
# Plot start ratio sweep
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ratios = list(start_ratio_results.keys())
barriers = [start_ratio_results[r]['metrics']['barrier'] for r in ratios]
keqs = [start_ratio_results[r]['metrics']['keq'] for r in ratios]

ax1.plot(ratios, barriers, 'o-', linewidth=2)
ax1.set_xlabel('Start Ratio')
ax1.set_ylabel('Barrier Height (kJ/mol)')
ax1.set_title('Barrier vs Equilibration')
ax1.grid(True, alpha=0.3)

ax2.plot(ratios, keqs, 's-', linewidth=2, color='orange')
ax2.set_xlabel('Start Ratio')
ax2.set_ylabel('Keq')
ax2.set_title('Keq vs Equilibration')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Mixed-State Analysis (EXP and HAM)

In [ ]:
# EXP Mixing
exp_params = {'beta': 1/300, 'C1': -300, 'C2': 0}

exp_results = analyzer.analyze_one_mixing_parameters(
    mixing_parameters=exp_params,
    temperature=310,
    method='EXP',
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)

exp_results['mixing_parameters'] = exp_params
exp_results['temperature'] = 310
exp_results['method'] = 'EXP'

analyzer.save_results(exp_results, 'FitEnergy/FreeEnergy_EXP.pkl')
print('EXP:', exp_results['metrics'])

In [ ]:
# HAM Mixing
ham_params = {'delta': 350, 'C1': -340, 'C2': 0}

ham_results = analyzer.analyze_one_mixing_parameters(
    mixing_parameters=ham_params,
    temperature=310,
    method='HAM',
    left_bound=LEFT_BOUND,
    right_bound=RIGHT_BOUND
)

ham_results['mixing_parameters'] = ham_params
ham_results['temperature'] = 310
ham_results['method'] = 'HAM'

analyzer.save_results(ham_results, 'FitEnergy/FreeEnergy_HAM.pkl')
print('HAM:', ham_results['metrics'])

In [ ]:
# Compare EXP and HAM
fig, ax = plt.subplots(figsize=(10, 6))

exp_pmf = exp_results['pmf'] - exp_results['pmf'].min()
ax.plot(exp_results['cv_values'], exp_pmf, 'b-', linewidth=2, label='EXP')

ham_pmf = ham_results['pmf'] - ham_results['pmf'].min()
ax.plot(ham_results['cv_values'], ham_pmf, 'r-', linewidth=2, label='HAM')

ax.set_xlabel('CV (Å)')
ax.set_ylabel('Free Energy (kJ/mol)')
ax.set_title('EXP vs HAM Mixing')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"EXP Barrier: {exp_results['metrics']['barrier']:.2f}, Keq: {exp_results['metrics']['keq']:.4f}")
print(f"HAM Barrier: {ham_results['metrics']['barrier']:.2f}, Keq: {ham_results['metrics']['keq']:.4f}")